# **Model Fine Tune with PyTorch**

Threre are several ways to do the fine-tune for a model. but we are considering only main two methods in this project.

They are:

* Fine-tune with all the parameters
* Fine-tune with the final layer only

In [ ]:
import torch
import torch.nn as nn 
from tqdm import tqdm
import math
import matplotlib.pyplot as plt
from urllib.request import urlopen
import pickle
import tarfile
import io
import os
import tempfile
from torch.utils.data import Dataset

### **Import Dataset**

In [6]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/35t-FeC-2uN1ozOwPs7wFg.gz"

In [10]:
url_open = urlopen(url)
#convert to tar file
tar_file = tarfile.open(fileobj= io.BytesIO(url_open.read()))
temp_file = tempfile.TemporaryDirectory()
tar_file.extractall(temp_file.name)
tar_file.close()

Define IMDB Dataset Class


In [ ]:
class IMDBDataset(Dataset):
    
    def __init__(self, root_dir, train = True):
        super().__init__()
        
        self.root_dir = os.path.join(root_dir,"train" if train else "test")
        self.negative_files = [os.path.join(self.root_dir, "neg",f) for f in os.listdir(os.path.join(self.root_dir,"neg")) if f.endswith('.txt')]
        self.positive_files = [os.path.join(self.root_dir,"pos",f) for f in os.listdir(os.path.join(self.root_dir,"pos")) if f.endswith(".txt")]
        
        self.files = self.negative_files + self.positive_files
        self.labels = [0] * len(self.negative_files) + [1] * len(self.positive_files)
        self.pos_idx = len(self.positive_files)
                

### **Word Embeddings**

In [ ]:
class WordEmbedding(nn.Module):
    
    def __init__(self, vocab_size, num_dims, dropout):
        super().__init__()
        self.embedding_layer = nn.Embedding(vocab_size, num_dims)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input_tokens):
        word_embed = self.embedding_layer(input_tokens)
        return word_embed